# Network Intrusion Detection

Corrected portfolio experiment. Run from top to bottom; previous metrics do not apply to this protocol.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data"
OUT = ROOT / "artifacts"
OUT.mkdir(exist_ok=True)
keras.utils.set_random_seed(42)
columns = (['duration','protocol_type','service','flag',
            'src_bytes','dst_bytes','land','wrong_fragment','urgent','hot'
,'num_failed_logins','logged_in','num_compromised','root_shell',
            'su_attempted','num_root','num_file_creations'
,'num_shells','num_access_files','num_outbound_cmds',
            'is_host_login','is_guest_login','count','srv_count','serror_rate'
,'srv_serror_rate','rerror_rate','srv_rerror_rate',
            'same_srv_rate','diff_srv_rate','srv_diff_host_rate',
            'dst_host_count','dst_host_srv_count'
,'dst_host_same_srv_rate','dst_host_diff_srv_rate',
            'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
            'dst_host_serror_rate'
,'dst_host_srv_serror_rate','dst_host_rerror_rate',
            'dst_host_srv_rerror_rate','outcome','level'])




In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
train = pd.read_csv(DATA / "KDDTrain+.txt", names=columns, header=None)
test = pd.read_csv(DATA / "KDDTest+.txt", names=columns, header=None)
assert train.shape[1] == 43 and test.shape[1] == 43
features = columns[:-2]
y = (train.outcome != "normal").astype(int)
y_test = (test.outcome != "normal").astype(int)
x_fit, x_val, y_fit, y_val = train_test_split(train[features], y, test_size=0.2, stratify=y, random_state=42)
categorical = ["protocol_type", "service", "flag"]
numeric = [c for c in features if c not in categorical]
preprocessor = ColumnTransformer([("numeric", StandardScaler(), numeric), ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)])
X_fit = preprocessor.fit_transform(x_fit).astype("float32")
X_val = preprocessor.transform(x_val).astype("float32")
X_test = preprocessor.transform(test[features]).astype("float32")

In [ ]:
model = keras.Sequential([keras.Input(shape=(X_fit.shape[1],)), keras.layers.Dense(128, activation="relu"), keras.layers.Dense(64, activation="relu"), keras.layers.Dense(32), keras.layers.Dense(1, activation="sigmoid")])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history = model.fit(X_fit, y_fit, validation_data=(X_val, y_val), epochs=30, batch_size=64, callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
baseline = LogisticRegression(max_iter=1000).fit(X_fit, y_fit)
results = {}
for name, scores in [("logistic_regression", baseline.predict_proba(X_test)[:, 1]), ("neural_network", model.predict(X_test, verbose=0).ravel())]:
    pred = scores >= 0.5
    results[name] = {"accuracy": accuracy_score(y_test, pred), "f1": f1_score(y_test, pred), "roc_auc": roc_auc_score(y_test, scores), "confusion_matrix": confusion_matrix(y_test, pred).tolist()}
print(json.dumps(results, indent=2))
(OUT / "metrics.json").write_text(json.dumps(results, indent=2))

In [ ]:
import joblib
import matplotlib.pyplot as plt
joblib.dump(preprocessor, OUT / "preprocessor.joblib")
joblib.dump(baseline, OUT / "baseline.joblib")
model.save(OUT / "model.keras")
pd.DataFrame(history.history)[["loss", "val_loss"]].plot(xlabel="Epoch", ylabel="Binary cross entropy", title="Training and validation loss")
plt.tight_layout()
plt.savefig(OUT / "training.png")